In [1]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2
%matplotlib qt

In [2]:
import os
os.chdir("..")


In [3]:
from notebooks.experimental import *

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import scipy
import napari

import osmnx as ox
ox.settings.use_cache = True
ox.settings.cache_folder = './osm_raw_cache'

from geo2stl import geo2stl as g2s
from geo2stl import sat2stl as s2s
from numpy2stl import numpy2stl as n2s
from city2stl import dem2stl as d2s
from city2stl import roads

In [5]:
Root = "C:/Users/eac84/Desktop/Desktop/FILES"


In [6]:
s2s.initialize_earth_engine()

## Cities

In [7]:
tags = {'building': True}

if 0:
	## Granada
	scale = .2
	center = (37.17827, -3.598) 
	dist = 1000

elif 1:
	## Philadelphia
	scale = .1
	center = (39.952, -75.16356) 
	dist = 2500

elif 0:
	## Cartegena 
	scale = .1
	center = (10.407, -75.545) 
	dist = 2500


In [8]:
TOLERANCE_M = 0.1            # Simplification tolerance in meters
ROAD_WIDTHS = {
	'motorway': 12, 'trunk': 10, 'primary': 8, 
	'secondary': 7, 'tertiary': 6, 'residential': 4, 'service': 2
}  

bbox = ox.utils_geo.bbox_from_point(center, dist)
bounds_NW = ((bbox[3],bbox[1]),(bbox[0],bbox[2]))
(N,S),(W,E) = bounds_NW


if 1:
	print("retriving data")
	dem = d2s.DEM(Root, (N,S,E,W))
	im = dem.data.copy()
	sat = s2s.fetch_bbox_image(N*1., S*1., E*1., W*1., 
				scale=2, dataset="esa")

	print("Retriving Data from OSMNX")
	gdf = ox.features_from_bbox(bbox, tags)
	gdf_zero = gdf.copy()

	road_graph = ox.graph_from_point(center, dist=dist, network_type='drive')
	gdf_roads = ox.graph_to_gdfs(road_graph, nodes=False)
	gdf_roads_zero = gdf_roads.copy()


if 1:
	im = dem.data.copy()
	for _ in range(3): im = scipy.ndimage.median_filter(im, 3)

	shape_out = im.shape
	outsize = np.array(shape_out)/max(shape_out)*100	
	im = transform.resize(im, outsize, anti_aliasing=True)	
	im = im - im.min() + im.ptp()*0.5

if 1:

	label = s2s.map_label_elevation(sat.copy() ,im, size=200)
	label = label.round(0)

if 1:
	gdf = fill_building_heights(gdf_zero.copy())
	gdf = reduce_buildings(gdf, TOLERANCE_M=dist/1000, a=3)
	gdf = add_building_z(gdf, label, bounds_NW)


if 1:
	gdf_roads2 = roads.get_road_segments(gdf_roads, ROAD_WIDTHS, TOLERANCE_M)
	polygon_list = polygon_list = list(gdf_roads2.geometry)
	gdf_roads2["z"] = roads.get_z_values(polygon_list, im, bounds_NW)
	gdf_roads2["z1"] = [x.mean() for x in gdf_roads2["z"]]	
	gdf_roads2["z0"] = 0
else:
	gdf_roads2 = None

retriving data
srtm_21_05
Retriving Data from OSMNX
Reducing buildings...


In [12]:
gdf

geometry  \
element id                                                              
node    367958244                          POINT (-75.15013 39.95091)   
        367958245                          POINT (-75.14456 39.94715)   
        367958246                          POINT (-75.14406 39.94684)   
        367958249                          POINT (-75.16879 39.94761)   
        367959475                          POINT (-75.18889 39.95778)   
...                                                               ...   
way     1446546940  POLYGON ((-75.1673 39.9461, -75.16749 39.94612...   
        1446546941  POLYGON ((-75.16729 39.94616, -75.16748 39.946...   
        1446546942  POLYGON ((-75.16727 39.94621, -75.16732 39.946...   
        1461247397  POLYGON ((-75.15398 39.95634, -75.15391 39.956...   
        1461258176  POLYGON ((-75.18824 39.95222, -75.1879 39.9521...   

                       addr:city addr:housenumber addr:postcode addr:state  \
element id                                                                   
node    367958244   Philadelphia              525         19106         PA   
        367958245            NaN              NaN           NaN         PA   
        367958246            NaN              NaN           NaN         PA   
        367958249            NaN              NaN           NaN         PA   
        367959475   Philadelphia              NaN         19104         PA   
...                          ...              ...           ...        ...   
way     1446546940  Philadelphia              340         19102         PA   
        1446546941  Philadelphia              338         19102         PA   
        1446546942  Philadelphia              336         19102         PA   
        1461247397           NaN              NaN           NaN        NaN   
        1461258176           NaN              NaN           NaN        NaN   

                           addr:street building  ele  \
element id                                             
node    367958244        Market Street      yes   10   
        367958245                  NaN      yes    6   
        367958246                  NaN      yes    6   
        367958249                  NaN      yes   12   
        367959475                  NaN      yes   23   
...                                ...      ...  ...   
way     1446546940  South Hicks Street    house  NaN   
        1446546941  South Hicks Street    house  NaN   
        1446546942  South Hicks Street    house  NaN   
        1461247397                 NaN      yes  NaN   
        1461258176                 NaN      yes  NaN   

                                          name         source  ...  \
element id                                                     ...   
node    367958244        Liberty Bell Pavilion  USGS Geonames  ...   
        367958245                  City Tavern  USGS Geonames  ...   
        367958246   Old Bookbinders Restaurant  USGS Geonames  ...   
        367958249    Colonial Dames of America  USGS Geonames  ...   
        367959475            Design Arts Annex  USGS Geonames  ...   
...                                        ...            ...  ...   
way     1446546940                         NaN            NaN  ...   
        1446546941                         NaN            NaN  ...   
        1446546942                         NaN            NaN  ...   
        1461247397                         NaN            NaN  ...   
        1461258176                         NaN            NaN  ...   

                   opening_date construction_date diet:meat  \
element id                                                    
node    367958244           NaN               NaN       NaN   
        367958245           NaN               NaN       NaN   
        367958246           NaN               NaN       NaN   
        367958249           NaN               NaN       NaN   
        367959475           NaN               NaN       NaN   
...            

In [9]:
gdf = add_building_z(gdf, label, bounds_NW)

In [15]:
gdf = None

In [13]:
gdf = fill_building_heights(gdf_zero.copy())
gdf = reduce_buildings(gdf, TOLERANCE_M=dist/1000, a=3)
gdf = add_building_z(gdf, label, bounds_NW)

print(len(gdf_zero), len(gdf))

Reducing buildings...
57109 5086


In [15]:
def render_2D(gdf, gdf_roads, im):

	x = np.array(gdf["z1"])
	x[0] = 0
	gdf["z1"] = x 
	im[0,0] = gdf["z1"].max()
	print(gdf["z1"].min(), gdf["z1"].max())

	fig, axs = plt.subplots(figsize=(15, 10), ncols=1, nrows=1, constrained_layout=True)# sharex=True, sharey=True, )

	axs = [axs]	
	axs[0].set_facecolor('#000000') # Dark background for a "pro" look

	# Plot Buildings colored by height
	if 1:
		gdf.plot(ax=axs[0], column='z1',cmap='jet', 
						alpha=0.9, edgecolor='#000', 
						linewidth=.5, legend=True, zorder=2)
	if 0:
		gdf_roads.plot(
			ax=axs[0], 
			column='z1', 
			cmap='jet', 
			alpha=0.9, 
			edgecolor='#FFFFFF', 
			linewidth=0,
			zorder=2
		)

	axs[0].imshow(im, cmap="jet", extent=np.array(bbox)[[0,2,1,3]])	



	

plt.close("all")
render_2D(gdf, gdf_roads2, label)

0.0 43.0


In [27]:
import napari


models = {}
models["landscape"] = get_landspace_model(label, bounds_NW, scale)
models["buildings"] = get_building_model(gdf, scale)
models["roads"] = get_building_model(gdf_roads2, scale)
#models["roads"] = roads.get_road_model(gdf_roads, scale )

#models["buildings2"] = get_building_model(gdf2, scale)
#models["terrain"] = get_landspace_model(label, bounds_NW, scale)
#models["space"] = get_bounds_model(gdf, scale)


Creating top...
Creating walls...
Creating bottom...
Simplifying Surfaces...
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
error 'triangles'
empty attempt recovery
attempt succeded
error 'triangles'
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded
empty attempt recovery
attempt succeded


In [ ]:
models = {}
label2 = scipy.ndimage.median_filter(label, 3)

models["landscape"] = get_landspace_model(label2, bounds_NW, scale*2)
plt.imshow(label2)

Creating top...
Creating walls...
Creating bottom...
Simplifying Surfaces...


: 

In [26]:
###################################
print("opening - Napari")
v = napari.current_viewer()
if v is None: v = napari.Viewer()
v.layers.clear()

for key in models:
	print(key)

	vertices, faces = models[key]
	print(len(faces) )
	surface = (vertices,faces)
	s = v.add_surface(surface)
	s.wireframe.visible = True#len(faces) < 20000
	s.name = key
	s.opacity = 0.8
	s.blend_mode = "translucent"

opening - Napari
landscape
21478


In [18]:
for key in models:
	print("-"*20)
	print(key)

	vertices, faces = models[key]
	print(vertices.mean(axis=0))
	print(vertices.ptp(axis=0))

	mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
	is_watertight = mesh.is_watertight
	is_volume = mesh.is_volume 
	print(is_watertight , is_volume)


--------------------
landscape
[ 3.99521469e+04 -7.51637066e+04  5.17711328e-01]
[44.67212124 58.3644637   4.3       ]
True True


In [ ]:
plt.close("all")
plt.figure()

vertices, faces = models["landscape"]
surfaces, normals = n2s.get_surfaces(vertices[faces].copy(), normals=None)
vx, new_faces = simplify_mesh_surfaces(vertices.copy(),faces.copy(), surfaces.copy())
models["landscape2"] = (vertices,new_faces)

print("---- Simplification complete ------")
print(len(new_faces), len(faces), (len(new_faces)/len(faces)))
open_edges = n2s.get_open_edges(new_faces)
len(open_edges) 

mesh = trimesh.Trimesh(vertices=vertices, faces=new_faces)
is_watertight = mesh.is_watertight
is_volume = mesh.is_volume 
print(is_watertight , is_volume)



NameError: name 'simplify_mesh_surfaces' is not defined

In [ ]:
%lprun -f reduce_surface simplify_mesh_surfaces(vertices.copy(),faces.copy(), surfaces.copy())

In [ ]:

print("opening - Napari")
v = napari.current_viewer()
if v is None: v = napari.Viewer()
v.layers.clear()

for key in models:
	print(key)

	vertices, faces = models[key]
	print(len(faces) )
	surface = (vertices,faces)
	s = v.add_surface(surface)
	s.wireframe.visible = True#len(faces) < 20000
	s.name = key
	s.opacity = 0.8
	s.blend_mode = "translucent"



In [ ]:
for key in models:
	print(key)

	vertices, faces = models[key]

	print(vertices.mean(axis=0))